
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.4_mla/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.4_mla/lab.ipynb)

# 3.4 Lab: Multi-Head Latent Attention (MLA)

GQA stores 8 × 128 = 1,024 values per token. Can we do better?

MLA compresses KV cache into a low-rank latent space, storing far fewer values per token while recovering full attention via matrix absorption.


In [ ]:
# Import
import torch
# Import
import time
# Import
import matplotlib.pyplot as plt
# Import
import numpy as np

# Config matching DeepSeek-V2 dimensions
device = "cuda" if torch.cuda.is_available() else "cpu"
# Print result to stdout
print(f"Device: {device}")

# Model dimensions
d_model = 5120        # hidden size
# Set n_heads
n_heads = 128         # num attention heads
# Set head_dim
head_dim = 128        # per-head dimension
# Set n_kv_heads_gqa
n_kv_heads_gqa = 8   # GQA group count
# Compute d_latent
d_latent = 512        # MLA latent dimension (much smaller than n_kv_heads * head_dim)


## Why GQA Still Wastes Memory

GQA-8 shares each KV head across 16 query heads, but each KV head still stores the full `head_dim=128` dimensions. For a sequence of length $L$:

$$\text{GQA KV bytes} = 2 \times L \times n_{kv} \times d_{head} \times 2 = 2 \times L \times 8 \times 128 \times 2$$

That's 4 KB per token. At 128K context, that's 512 MB per layer just for KV cache.


In [ ]:
# Define gqa_kv_bytes function
def gqa_kv_bytes(seq_len, n_kv_heads=8, head_dim=128, dtype_bytes=2):
    """KV cache size for GQA: 2 (K+V) * seq * heads * dim * dtype"""
    # Return the computed result
    return 2 * seq_len * n_kv_heads * head_dim * dtype_bytes

# Show the problem at various sequence lengths
seq_lengths = [1024, 4096, 16384, 65536, 131072]
# Iterate over
for s in seq_lengths:
    # Compute mb
    mb = gqa_kv_bytes(s) / (1024**2)
    # Print result to stdout
    print(f"  seq_len={s:>7,}: GQA-8 KV = {mb:>8.1f} MB per layer")


## MLA: Low-Rank KV Compression

Instead of storing `n_kv_heads × head_dim = 1024` values per token, MLA stores a single latent vector of dimension `d_latent=512`:

$$c_t = W_{dkv} \cdot x_t \quad \text{where } W_{dkv} \in \mathbb{R}^{d_{latent} \times d_{model}}$$

At inference, we store only $c_t$ (512 values) instead of full KV pairs (1024 values per GQA, or 32768 for MHA).

To recover keys and values for attention: $K_t = W_{UK} \cdot c_t$, $V_t = W_{UV} \cdot c_t$


In [ ]:
# Build the MLA projection matrices
torch.manual_seed(42)

# Down-projection: d_model -> d_latent (compress)
W_dkv = torch.randn(d_latent, d_model, device=device, dtype=torch.float16) * 0.01

# Up-projections: d_latent -> full KV space (decompress)
W_UK = torch.randn(n_heads, head_dim, d_latent, device=device, dtype=torch.float16) * 0.01
W_UV = torch.randn(n_heads, head_dim, d_latent, device=device, dtype=torch.float16) * 0.01

print(f"W_dkv shape: {W_dkv.shape}  (compress {d_model} -> {d_latent})")
print(f"W_UK shape:  {W_UK.shape}  (decompress latent -> K per head)")
print(f"W_UV shape:  {W_UV.shape}  (decompress latent -> V per head)")

# Simulate compressing a batch of tokens
batch_seq = 4096
x = torch.randn(batch_seq, d_model, device=device, dtype=torch.float16)

# Compress: store only this in KV cache
c = x @ W_dkv.T  # [seq, d_latent]
print(f"\nInput tokens: {x.shape} -> Latent cache: {c.shape}")
print(f"Compression ratio: {d_model}/{d_latent} = {d_model/d_latent:.1f}x fewer dims stored")


In [ ]:
# Define mla_kv_bytes function
def mla_kv_bytes(seq_len, d_latent=512, dtype_bytes=2):
    """MLA stores one latent vector per token (K and V combined)"""
    # Return the computed result
    return seq_len * d_latent * dtype_bytes

# Compare GQA-8 vs MLA across sequence lengths
seq_range = np.array([512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072])
# Compute gqa_mb
gqa_mb = np.array([gqa_kv_bytes(s) / (1024**2) for s in seq_range])
# Compute mla_mb
mla_mb = np.array([mla_kv_bytes(s) / (1024**2) for s in seq_range])

fig_3, ax_3 = plt.subplots(1, 1, figsize=(9, 5))
ax_3.plot(seq_range, gqa_mb, 'o-', color='#e74c3c', linewidth=2, label='GQA-8 (8×128 = 1024 vals/token)')
ax_3.plot(seq_range, mla_mb, 's-', color='#2ecc71', linewidth=2, label=f'MLA (d_latent={d_latent} vals/token)')
ax_3.fill_between(seq_range, mla_mb, gqa_mb, alpha=0.15, color='#2ecc71')
ax_3.set_xscale('log', base=2)
ax_3.set_yscale('log', base=2)
ax_3.set_xlabel('Sequence Length', fontsize=12)
ax_3.set_ylabel('KV Cache per Layer (MB)', fontsize=12)
ax_3.set_title('KV Cache Memory: GQA-8 vs MLA', fontsize=14)
ax_3.legend(fontsize=11)
ax_3.grid(True, alpha=0.3)
# Annotate savings at 128K
savings = (1 - mla_mb[-1]/gqa_mb[-1]) * 100
ax_3.annotate(f'{savings:.0f}% smaller', xy=(131072, mla_mb[-1]), fontsize=11,
            # Compute xytext
            xytext=(131072, (gqa_mb[-1]+mla_mb[-1])/2), ha='center', color='#27ae60', fontweight='bold')
# Adjust spacing between subplots
plt.tight_layout()
# Render the figure
plt.show()
# Print result to stdout
print(f"At 128K tokens: GQA-8 = {gqa_mb[-1]:.1f} MB, MLA = {mla_mb[-1]:.1f} MB ({savings:.0f}% reduction)")


## GPU-Measured Memory: Allocate and Compare

Let's actually allocate the KV cache tensors on GPU and measure real memory usage.


In [ ]:
# Conditional check
if device == "cuda":
    # Free unused GPU memory
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    # Set seq_test
    seq_test = 16384
    # Set n_layers
    n_layers = 60  # DeepSeek-V2 has 60 layers

    # Measure GQA allocation
    torch.cuda.reset_peak_memory_stats()
    # Compute gqa_cache
    gqa_cache = torch.zeros(n_layers, 2, seq_test, n_kv_heads_gqa, head_dim, device=device, dtype=torch.float16)
    # Compute gqa_mem
    gqa_mem = torch.cuda.max_memory_allocated() / (1024**3)
    # Free memory by deleting tensors
    del gqa_cache
    # Free unused GPU memory
    torch.cuda.empty_cache()

    # Measure MLA allocation
    torch.cuda.reset_peak_memory_stats()
    # Compute mla_cache
    mla_cache = torch.zeros(n_layers, seq_test, d_latent, device=device, dtype=torch.float16)
    # Compute mla_mem
    mla_mem = torch.cuda.max_memory_allocated() / (1024**3)
    # Free memory by deleting tensors
    del mla_cache
    # Free unused GPU memory
    torch.cuda.empty_cache()

    # Print result to stdout
    print(f"Seq length: {seq_test:,} | Layers: {n_layers}")
    # Print result to stdout
    print(f"  GQA-8 KV cache: {gqa_mem:.2f} GB")
    # Print result to stdout
    print(f"  MLA KV cache:   {mla_mem:.2f} GB")
    # Print result to stdout
    print(f"  Savings:        {(1-mla_mem/gqa_mem)*100:.1f}%")
else:
    # Print result to stdout
    print("No GPU available -- showing theoretical calculation only")
    # Set seq_test
    seq_test = 16384
    # Set n_layers
    n_layers = 60
    # Compute gqa_theoretical
    gqa_theoretical = n_layers * gqa_kv_bytes(seq_test) / (1024**3)
    # Compute mla_theoretical
    mla_theoretical = n_layers * mla_kv_bytes(seq_test) / (1024**3)
    # Print result to stdout
    print(f"  GQA-8 theoretical: {gqa_theoretical:.2f} GB")
    # Print result to stdout
    print(f"  MLA theoretical:   {mla_theoretical:.2f} GB")


## The Matrix Absorption Trick

Naive MLA decompresses $K = W_{UK} \cdot c$ before computing attention. This adds a matmul per layer per token.

**Absorption** folds $W_{UK}$ into the query projection instead:

$$\text{score} = q \cdot K^T = (q \cdot W_{UK}) \cdot c^T = \tilde{q} \cdot c^T$$

We pre-compute $\tilde{q} = q \cdot W_{UK}$ once, then attention operates directly on the compressed latents. No decompression needed at decode time.


In [ ]:
# Benchmark: naive decompress vs absorbed attention
seq_len_bench = 8192
# Set batch
batch = 1

# Simulated cached latents and query
c_cache = torch.randn(batch, seq_len_bench, d_latent, device=device, dtype=torch.float16)
# Compute q
q = torch.randn(batch, n_heads, 1, head_dim, device=device, dtype=torch.float16)  # single decode step

# W_UK reshaped for batched matmul: [n_heads, head_dim, d_latent]
W_UK_bench = torch.randn(n_heads, head_dim, d_latent, device=device, dtype=torch.float16) * 0.01

# Define naive_attention function
def naive_attention():
    """Decompress K from latents, then compute attention scores"""
    # Decompress: [batch, seq, d_latent] @ [n_heads, d_latent, head_dim] -> need reshape
    K_full = torch.einsum('bsd,nhd->bsnh', c_cache, W_UK_bench.transpose(1,2))  # [b, seq, heads, head_dim]
    # Compute K_full
    K_full = K_full.permute(0, 2, 1, 3)  # [b, heads, seq, head_dim]
    # Compute scores
    scores = torch.matmul(q, K_full.transpose(-2, -1))  # [b, heads, 1, seq]
    # Return the computed result
    return scores

# Define absorbed_attention function
def absorbed_attention():
    """Absorb W_UK into query, attend directly on compressed latents"""
    # q: [b, heads, 1, head_dim], W_UK: [heads, head_dim, d_latent]
    q_absorbed = torch.einsum('bhqd,hdc->bhqc', q, W_UK_bench)  # [b, heads, 1, d_latent]
    # Attend on compressed cache directly
    c_expanded = c_cache.unsqueeze(1).expand(-1, n_heads, -1, -1)  # [b, heads, seq, d_latent]
    # Compute scores
    scores = torch.matmul(q_absorbed, c_expanded.transpose(-2, -1))  # [b, heads, 1, seq]
    # Return the computed result
    return scores

# Warmup
if device == "cuda":
    # Iterate over
    for _ in range(10):
        naive_attention()
        absorbed_attention()
    # Wait for GPU ops to finish
    torch.cuda.synchronize()

# Time naive
n_iters = 100
# Conditional check
if device == "cuda":
    # Wait for GPU ops to finish
    torch.cuda.synchronize()
    # Compute t0
    t0 = time.perf_counter()
    # Iterate over
    for _ in range(n_iters):
        naive_attention()
    # Wait for GPU ops to finish
    torch.cuda.synchronize()
    # Compute naive_ms
    naive_ms = (time.perf_counter() - t0) / n_iters * 1000

    # Time absorbed
    torch.cuda.synchronize()
    # Compute t0
    t0 = time.perf_counter()
    # Iterate over
    for _ in range(n_iters):
        absorbed_attention()
    # Wait for GPU ops to finish
    torch.cuda.synchronize()
    # Compute absorbed_ms
    absorbed_ms = (time.perf_counter() - t0) / n_iters * 1000

    # Print result to stdout
    print(f"Seq length: {seq_len_bench:,} | Heads: {n_heads} | Latent: {d_latent}")
    # Print result to stdout
    print(f"  Naive (decompress + attend):  {naive_ms:.3f} ms")
    # Print result to stdout
    print(f"  Absorbed (attend on latent):  {absorbed_ms:.3f} ms")
    # Print result to stdout
    print(f"  Speedup: {naive_ms/absorbed_ms:.2f}x")
else:
    # Print result to stdout
    print("GPU required for timing benchmark -- run on Colab/Molab")


## Key Takeaways

| | GQA-8 | MLA (d=512) |
|---|---|---|
| Values stored per token | 1,024 | 512 |
| Cache at 128K (per layer) | 512 MB | 128 MB |
| Decode attention | Standard matmul | Absorbed (no decompress) |
| Tradeoff | Simple, proven | Extra projection weights, training complexity |

MLA achieves ~50-75% KV cache reduction while maintaining attention quality through low-rank compression. The absorption trick eliminates the decompression cost at decode time, making MLA faster *and* smaller than GQA for long-context inference.
